In [3]:
import sys
!{sys.executable} -m pip freeze > requirements.txt

In [4]:
import os
print(os.getcwd())

C:\Jupyter Notebook


In [5]:
pip install fastapi uvicorn

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 771.3 kB/s eta 0:00:02
   ---------- ----------------------------- 0.5/2.0 MB 771.3 kB/s eta 0:00:02
   --------------- ------------------------ 0.8/2.0 MB 685.6 kB/s eta 0:00:02
   --------------- ------------------------ 0.8/2.0 MB 685.6 kB/s eta 0:00:02
   --------------- ------------------------ 0.8/2.0 MB 685.6 kB/s eta 0:00:02
   -------------------- ------------------- 1.0/2.0 MB 598.9 kB/s eta 0:00:02
   ------------------------- -------------- 1.3/2.0 MB 678.8 kB/s eta 0:00:02
   ------------------------------ --------- 1.6/2.0 MB 748.0 kB/s eta 0:00:01
   ------------------------------ --------- 1.6/2.0 MB 748.0 kB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.0 MB 736.0 kB/s eta 0:00:01
   ----------

In [6]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [14]:
%%writefile crudAPI_with_SQL_Database.py

from fastapi import FastAPI
from pydantic import BaseModel
from sqlalchemy import create_engine,Column,Interger,String,Boolean
from sqlalchemy.orm import declarative_base,sessionmaker

app=FastAPI()

DATABASE_URL="sqlite:///./todo.db"

engine=create_engine(DATABASE_URL,connect_args={"check_same_thread":False})

Base=declarative_base()

Sessionlocal=sessionmaker(autocommit=False,autoflush=-False,bind=engine)

class TaskDB(Base):
    __tablename__="tasks"
    id=Column(Integer,primary_key=True)
    title=Column(String)
    is_done=Column(Boolean, default=False)
    Base.metadata.create_all(bind_engine)

class Task(BaseModel):
    id:int
    title:string
    is_done:bool=False

@app.post("/tasks")
def create_task(task:Task):
    db=SessionLocal()
    new_task=TaskDB(id=task.id,title=task.string,is_done=task.is_done)
    db.add(new_task)
    db.commit()
    db.refresh(new_task)
    db.close()
    return new_task

@app.get("/tasks")
def get_task():
    db=SessionLocal()
    tasks=db.query(TaskDB).all()
    return tasks

@app.put("/tasks/{task_id}")
def update_task(task_id: int, updated_task: Task):
    db = SessionLocal()
    task = db.query(TaskDB).filter(TaskDB.id == task_id).first()
    if task:
        task.title = updated_task.title
        task.is_done = updated_task.is_done
        db.commit()
        db.refresh(task)
        db.close()
        return task
    db.close()
    return {"error": "Task not found"}


@app.delete("/tasks/{task_id}")
def delete_task(task_id: int):
    db = SessionLocal()
    task = db.query(TaskDB).filter(TaskDB.id == task_id).first()
    if task:
        db.delete(task)
        db.commit()
        db.close()
        return {"message": "Task deleted!"}
    db.close()
    return {"error": "Task not found"}



    

    


Writing crud.py
